# Set Up (this notebook trains a final model before evaluating on test)

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import os
import gc
import random
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

In [ ]:
import sys
from pathlib import Path

# Get the directory of the current notebook (gnn_training_pipeline)
current_dir = Path.cwd()

# Get the parent directory (QuantNets)
parent_dir = current_dir.parent

# Add QuantNets to the Python path
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

# Import from the gnn folder
from gnn.architectures_inject import GATv2ConvNet as GATv2ConvNet_Inject
from gnn.architectures_normal import GATv2ConvNet as GATv2ConvNet_Normal

from metrics import calc_r2_corr

# Configuration

In [ ]:
# --- Determinism ---
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Configuration ---
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'

gnn_dir = base_dir / 'models_gnn_dnanexus'
results_dir = gnn_dir / 'gnn_scaling_law_results'
predictions_dir = gnn_dir / 'gnn_predictions_scaling_law'
plots_dir = gnn_dir / 'gnn_scaling_law_plots'

for d in [results_dir, predictions_dir, plots_dir]:
    d.mkdir(parents=True, exist_ok=True)

targets = {
    'GF': ('GF', 'p20016_i2'),
    'PAL': ('PAL', 'p20197_i2'),
    'DSST': ('DSST', 'p23324_i2'),
    'TMT': ('TMT', 'p6350_i2'),
}

# (Nodes count isn't strictly needed for definition, but good to know)
data_configs = {
    'FC25': {'nodes': 21},
    'FC100': {'nodes': 55},
    'MIND': {'nodes': 68},
}

sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAX_EPOCHS = 100
WARMUP_EPOCHS = 10
EARLY_STOP_PATIENCE = 20
N_TRIALS = 50
N_SPLITS = 10

In [ ]:
# Helper from process_graph_pipeline/csv_to_graph.py
def parse_region_map(region_file_path):
    with open(region_file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    raw_pairs = []
    for line in lines:
        if line.startswith("FC"):
            inner = line[line.find("(")+1:line.find(")")]
            src, dst = inner.split("-")
        else:
            src, dst = line.split("-")
        raw_pairs.append((src, dst))

    unique_nodes = []
    for src, dst in raw_pairs:
        if src not in unique_nodes: unique_nodes.append(src)
        if dst not in unique_nodes: unique_nodes.append(dst)
        
    node_to_idx = {name: idx for idx, name in enumerate(unique_nodes)}
    edge_list = [(node_to_idx[src], node_to_idx[dst]) for src, dst in raw_pairs]
    
    return len(unique_nodes), edge_list, lines


class BrainGraphDataset(torch.utils.data.Dataset):
    def __init__(self, X_edge_features, demographics_features, y_targets, eids, 
                 num_nodes, mapping_edges, sparsity_percent):
        """
        X_edge_features: NumPy array of flattened edges (num_samples, num_edges)
        demographics_features: NumPy array of patient traits (num_samples, num_demo_feats) 
        y_targets: target regression array
        eids: standard subjects 
        sparsity_percent: float from 0.0 to 1.0 (1.0 = keep all edges)
        """
        self.X_edges = torch.tensor(X_edge_features, dtype=torch.float)
        self.X_demo = torch.tensor(demographics_features, dtype=torch.float)
        self.targets = torch.tensor(y_targets, dtype=torch.float)
        self.eids = eids
        self.num_nodes = num_nodes
        self.num_total_edges = len(mapping_edges)
        
        self.k_edges_to_keep = max(1, int(np.floor(sparsity_percent * self.num_total_edges)))

        self.src_global = np.array([e[0] for e in mapping_edges])
        self.dst_global = np.array([e[1] for e in mapping_edges])
        
    def __len__(self):
        return len(self.eids)

    def __getitem__(self, idx):
        eid = self.eids.iloc[idx] if hasattr(self.eids, 'iloc') else self.eids[idx]
        y = self.targets[idx]
        w_u = self.X_edges[idx].numpy()
        
        if self.k_edges_to_keep < self.num_total_edges:
            top_idx = np.argpartition(np.abs(w_u), -self.k_edges_to_keep)[-self.k_edges_to_keep:]
        else:
            top_idx = np.arange(self.num_total_edges)

        w_selected = w_u[top_idx]
        src_u = self.src_global[top_idx]
        dst_u = self.dst_global[top_idx]
        
        src = np.concatenate([src_u, dst_u])
        dst = np.concatenate([dst_u, src_u])
        w = np.concatenate([w_selected, w_selected])
        edge_index = np.stack([src, dst])
        
        # Graph components setup
        x = torch.arange(self.num_nodes, dtype=torch.long) # Node indices
        demographics = self.X_demo[idx].view(1, -1)        # 1xD Demo array
        
        return Data(
            x=x, 
            edge_index=torch.tensor(edge_index, dtype=torch.long), 
            edge_attr=torch.tensor(w, dtype=torch.float),
            demographics=demographics,
            y=y.view(1),
            eid=eid
        )

# Optuna and analysis functions

In [ ]:
def train_gnn(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            out = model(data)
            loss = criterion(out, data.y.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_gnn(model, loader, criterion):
    model.eval()
    total_loss = 0
    preds, actuals, eids = [], [], []
    for data in loader:
        data = data.to(device)
        with autocast(device_type='cuda'):
            out = model(data)
            loss = criterion(out, data.y.view(-1))
        total_loss += loss.item() * data.num_graphs
        
        preds.extend(out.cpu().float().view(-1).numpy())
        actuals.extend(data.y.cpu().view(-1).numpy())
        eids.extend(data.eid.cpu().numpy())
    return (total_loss / len(loader.dataset)), np.array(preds), np.array(actuals), np.array(eids)

In [ ]:
def get_optimizer_params(model, weight_decay):
    no_decay = ['bias', 'batch_norms'] # Add names of modules/parameters that shouldn't decay
    
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    return optimizer_grouped_parameters

In [ ]:
def objective(trial, X_brain_train, X_demo_train, y_train, eids_train, 
              X_brain_val, X_demo_val, y_val, eids_val, 
              num_nodes, mapping_edges):
    
    # Suggest hyperparameters (including architecture)
    architecture = trial.suggest_categorical('architecture', ['normal', 'inject'])

    hidden_channels = trial.suggest_int('hidden_channels', 4, 7)
    hidden_channels = 2 ** hidden_channels
    embedding_dim = trial.suggest_int('embedding_dim', 3, 5)
    embedding_dim = 2 ** embedding_dim
    gnn_num_layers = trial.suggest_int('gnn_num_layers', 2, 4)
    classifier_num_layers = trial.suggest_int('classifier_num_layers', 1, 4)

    hidden_heads = trial.suggest_int('hidden_heads', 0, 3) 
    hidden_heads = 2 ** hidden_heads
    demo_embed_dim = trial.suggest_int('demo_embed_dim', 3, 5)
    demo_embed_dim = 2 ** demo_embed_dim

    edge_dropout = trial.suggest_float('edge_dropout', 0.0, 0.4, step=0.05)
    node_dropout = trial.suggest_float('node_dropout', 0.0, 0.6, step=0.05)
    classifier_dropout = trial.suggest_float('classifier_dropout', 0.0, 0.8, step=0.05)
    concat = trial.suggest_categorical('concat', [True, False])
    include_jk = trial.suggest_categorical('include_jk', [True, False])
    
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    sparsity_percent = trial.suggest_float('sparsity_percent', 0.05, 1.0, step=0.05) 
    
    train_dataset = BrainGraphDataset(X_brain_train, X_demo_train, y_train, eids_train, num_nodes, mapping_edges, sparsity_percent)
    val_dataset = BrainGraphDataset(X_brain_val, X_demo_val, y_val, eids_val, num_nodes, mapping_edges, sparsity_percent)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    num_demo_features = X_demo_train.shape[1] 
    
    # Select the model class based on Optuna's suggestion
    ModelClass = GATv2ConvNet_Inject if architecture == 'inject' else GATv2ConvNet_Normal

    # Initialize new dynamic model
    model = ModelClass(
        out_dim=1, 
        in_channels=num_nodes,          
        gnn_num_layers=gnn_num_layers, 
        classifier_num_layers=classifier_num_layers,
        hidden_channels=hidden_channels, 
        embedding_dim=embedding_dim,
        demo_dim=num_demo_features,
        edge_dropout_rate=edge_dropout,
        node_dropout_rate=node_dropout,
        hidden_heads=hidden_heads,
        demo_embed_dim=demo_embed_dim,
        include_demo=True,
        classifier_dropout_rate=classifier_dropout,
        concat=concat,
        include_jk=include_jk
    ).to(device)
    
    param_groups = get_optimizer_params(model, weight_decay)
    optimizer = torch.optim.AdamW(param_groups, lr=lr)
    criterion = nn.MSELoss()

    warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
    decay_epochs = MAX_EPOCHS - WARMUP_EPOCHS
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=decay_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])

    best_val_loss = float('inf')
    best_epoch = 0
    early_stop_patience = EARLY_STOP_PATIENCE
    no_improve = 0
    
    scaler = GradScaler()
    for epoch in tqdm(range(MAX_EPOCHS), desc="Training Epochs", leave=False):
        train_gnn(model, train_loader, optimizer, criterion, scaler)
        scheduler.step()
        val_loss, _, _, _ = eval_gnn(model, val_loader, criterion)
        
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1 
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= early_stop_patience:
                break
                
    trial.set_user_attr('best_epoch', best_epoch)
    
    return best_val_loss

In [ ]:
def gnn_analysis(df, data_name, target_name, sample_size, n_splits=10):
    # Grab the relevant properties for the mapping from data_configs
    regions_file = data_configs[data_name][0]
    demographic_cols = data_configs[data_name][1]
    
    # If there is no graph structure (e.g. 'demo' only), we skip the GNN 
    if regions_file is None:
        print(f"  Skipping {data_name}: GNN requires graph structure.")
        return None

    # Parse structural nodes & edges
    num_nodes, mapping_edges, brain_cols = parse_region_map(regions_file)
    
    eids = df.index if 'eid' not in df.columns else df['eid']
    y = df[targets[target_name][1]]
    
    # Figure out categorical vs numerical bounds for the demographic pipeline
    cat_vars = [c for c in ['sex', 'assessment_centre', 'p31', 'p54_i2'] if c in demographic_cols]
    num_vars = [c for c in demographic_cols if c not in cat_vars]

    outer_cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    outer_mae, outer_rmse, outer_r2, outer_r2_corr = [], [], [], []
    
    target_predictions_dir = predictions_dir / target_name
    target_predictions_dir.mkdir(parents=True, exist_ok=True)
    preds_path = target_predictions_dir / f'GNN_preds_{data_name}_{target_name}_{sample_size}.csv'

    # Filter available structural and demographic columns respectively 
    actual_brain_cols = [c for c in brain_cols if c in df.columns]
    X_brain_raw = df[actual_brain_cols].fillna(0.0) 
    X_demo_raw = df[demographic_cols]

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(eids, y), start=1):
        # Inner Split parameters mapping (using 15% inner validation as requested)
        inner_train_idx_rel, inner_val_idx_rel = train_test_split(
            np.arange(len(train_idx)), test_size=0.15, random_state=seed
        )
        inner_train_idx = train_idx[inner_train_idx_rel]
        inner_val_idx = train_idx[inner_val_idx_rel]

        # --- 1A. Inner Loop Scaling ---
        inner_demo_preprocessor = ColumnTransformer(transformers=[
            ('num', StandardScaler(), num_vars),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_vars)
        ])
        inner_edge_preprocessor = StandardScaler()
        inner_y_preprocessor = StandardScaler()

        # Fit & Transform ONLY on Inner Train
        X_brain_train = inner_edge_preprocessor.fit_transform(X_brain_raw.iloc[inner_train_idx])
        X_demo_train = inner_demo_preprocessor.fit_transform(X_demo_raw.iloc[inner_train_idx])
        y_train = inner_y_preprocessor.fit_transform(y.iloc[inner_train_idx].values.reshape(-1, 1)).flatten()
        eids_train = eids.iloc[inner_train_idx]

        # Transform Inner Val
        X_brain_val = inner_edge_preprocessor.transform(X_brain_raw.iloc[inner_val_idx])
        X_demo_val = inner_demo_preprocessor.transform(X_demo_raw.iloc[inner_val_idx])
        y_val = inner_y_preprocessor.transform(y.iloc[inner_val_idx].values.reshape(-1, 1)).flatten()
        eids_val = eids.iloc[inner_val_idx]
        
        # --- 1B. Outer Loop Scaling ---
        demo_preprocessor = ColumnTransformer(transformers=[
            ('num', StandardScaler(), num_vars),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_vars)
        ])
        edge_preprocessor = StandardScaler()
        y_preprocessor = StandardScaler()
        
        # Construct exact scaled Outer sets (Fit on FULL 90% outer train data)
        X_brain_train_outer = edge_preprocessor.fit_transform(X_brain_raw.iloc[train_idx])
        X_demo_train_outer = demo_preprocessor.fit_transform(X_demo_raw.iloc[train_idx])
        y_train_outer = y_preprocessor.fit_transform(y.iloc[train_idx].values.reshape(-1, 1)).flatten()
        eids_train_outer = eids.iloc[train_idx]
        
        # Transform strict Test Set
        X_brain_test = edge_preprocessor.transform(X_brain_raw.iloc[test_idx])
        X_demo_test = demo_preprocessor.transform(X_demo_raw.iloc[test_idx])
        y_test_scaled = y_preprocessor.transform(y.iloc[test_idx].values.reshape(-1, 1)).flatten()
        eids_test = eids.iloc[test_idx]
        
        # --- 2. PyG Network Sub-Fold Optuna Optimization --- 
        # Using lambda allows us to bypass global bindings cleanly
        objective_func = lambda trial: objective(
            trial, X_brain_train, X_demo_train, y_train, eids_train, 
            X_brain_val, X_demo_val, y_val, eids_val, 
            num_nodes, mapping_edges
        )
        
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        pruner = optuna.pruners.HyperbandPruner(min_resource=50, max_resource=MAX_EPOCHS, reduction_factor=3)
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed+fold), pruner=pruner)
        
        # NOTE: GNNs execute much slower than XGBoost trees. Using 50 trials is highly recommended for time
        study.optimize(objective_func, n_trials=N_TRIALS, show_progress_bar=True)

        # Output graphs cleanly
        fold_plot_dir = plots_dir / target_name / data_name / f'n_{sample_size}' / f'fold_{fold}'
        fold_plot_dir.mkdir(parents=True, exist_ok=True)
        fig_history = plot_optimization_history(study)
        fig_history.write_image(str(fold_plot_dir / 'optimization_history.png'))
        fig_history.write_html(str(fold_plot_dir / 'optimization_history.html'))
        fig_parallel = plot_parallel_coordinate(study)
        fig_parallel.write_image(str(fold_plot_dir / 'parallel_coordinate.png'))
        fig_parallel.write_html(str(fold_plot_dir / 'parallel_coordinate.html'))
        fig_importances = plot_param_importances(study)
        fig_importances.write_image(str(fold_plot_dir / 'param_importances.png'))
        fig_importances.write_html(str(fold_plot_dir / 'param_importances.html'))

        # --- 3. Retrain utilizing optimal parameters for Final Model ---
        best_trial = study.best_trial
        best_params = study.best_trial.params

        best_epoch = best_trial.user_attrs['best_epoch']
        
        train_full_dataset = BrainGraphDataset(
            X_brain_train_outer, X_demo_train_outer, y_train_outer, eids_train_outer, 
            num_nodes, mapping_edges, best_params['sparsity_percent']
        )
        test_dataset = BrainGraphDataset(
            X_brain_test, X_demo_test, y_test_scaled, eids_test,
            num_nodes, mapping_edges, best_params['sparsity_percent']
        )
        
        train_full_loader = DataLoader(train_full_dataset, batch_size=best_params['batch_size'], shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

        # Select the best model architecture found during tuning
        best_architecture = best_params['architecture']
        FinalModelClass = GATv2ConvNet_Inject if best_architecture == 'inject' else GATv2ConvNet_Normal

        final_model = FinalModelClass(
            out_dim=1,
            in_channels=num_nodes,
            gnn_num_layers=best_params['gnn_num_layers'],
            classifier_num_layers=best_params['classifier_num_layers'],
            hidden_channels= (2 ** best_params['hidden_channels']),
            embedding_dim= (2 ** best_params['embedding_dim']),
            demo_dim=X_demo_train_outer.shape[1],
            edge_dropout_rate=best_params['edge_dropout'],
            node_dropout_rate=best_params['node_dropout'],
            hidden_heads= (2 ** best_params['hidden_heads']),
            demo_embed_dim= (2 ** best_params['demo_embed_dim']),
            include_demo=True,
            classifier_dropout_rate=best_params['classifier_dropout'],
            concat=best_params['concat'],
            include_jk=best_params['include_jk']
        ).to(device)
        
        param_groups = get_optimizer_params(final_model, best_params['weight_decay'])
        optimizer = torch.optim.AdamW(param_groups, lr=best_params['lr'])
        criterion = nn.MSELoss()

        warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        decay_epochs = MAX_EPOCHS - WARMUP_EPOCHS
        cosine_scheduler = CosineAnnealingLR(optimizer, T_max=decay_epochs)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])

        scaler = GradScaler()
        for _ in range(best_epoch): 
            train_gnn(final_model, train_full_loader, optimizer, criterion, scaler)
            scheduler.step()
            
        _, preds_scaled, actuals_scaled, test_eids = eval_gnn(final_model, test_loader, criterion)
        
        # Unstandardize predictions and actuals back to original scale
        preds = y_preprocessor.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
        actuals = y_preprocessor.inverse_transform(actuals_scaled.reshape(-1, 1)).flatten()

        fold_df = pd.DataFrame({'fold': fold, 'eid': test_eids, 'actual': actuals, 'predicted': preds})
        fold_df.to_csv(preds_path, mode='a', header=not preds_path.exists(), index=False)

        mae = mean_absolute_error(actuals, preds)
        rmse = np.sqrt(mean_squared_error(actuals, preds))
        r2 = r2_score(actuals, preds)
        r2_corr = calc_r2_corr(actuals, preds)

        outer_mae.append(mae)
        outer_rmse.append(rmse)
        outer_r2.append(r2) 
        outer_r2_corr.append(r2_corr)
        print(f'  Fold {fold:02d} • MAE={mae:.3f} • RMSE={rmse:.3f} • R²={r2:.3f} • R²(corr)={r2_corr:.3f}')

    return {
        'mean_mae': np.mean(outer_mae), 'std_mae': np.std(outer_mae),
        'mean_rmse': np.mean(outer_rmse), 'std_rmse': np.std(outer_rmse),
        'mean_r2': np.mean(outer_r2), 'std_r2': np.std(outer_r2),
        'mean_r2_corr': np.mean(outer_r2_corr), 'std_r2_corr': np.std(outer_r2_corr),
    }


# Scaling Law Training Loop

In [ ]:
for target_name, (test_key, score_col) in targets.items():
    print(f'\n{"="*60}\nTARGET: {target_name}\n{"="*60}')
    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    target_splits_dir = splits_dir / target_name
    df_full = pd.read_csv(data_file)
    if 'eid' in df_full.columns:
        df_full.set_index('eid', inplace=True)
        
    for data_name in data_configs.keys():
        print(f'\n--- {target_name} vs. {data_name} ---')
        for sample_size in sample_sizes:
            eid_file = (target_splits_dir / f'{target_name}_all_eids.txt') if sample_size == 'all' else (target_splits_dir / f'{target_name}_eids_{sample_size}.txt')
            if not eid_file.exists(): continue
            
            sample_eids = np.loadtxt(eid_file, dtype=int)
            df = df_full[df_full.index.isin(sample_eids)]
            actual_n = len(df)
            
            print(f'\n[{target_name} | {data_name} | n={sample_size} ({actual_n} rows)]')
            start_time = time.time()
            metrics = gnn_analysis(df, data_name, target_name, sample_size, n_splits=N_SPLITS)
            elapsed = time.time() - start_time
            print(f'  Time: {elapsed:.2f}s')

            results_file = results_dir / f'gnn_scaling_law_results_{target_name}.csv'
            row_df = pd.DataFrame([{'target_name': target_name, 'data_name': data_name, 'sample_size': sample_size, 'actual_n': actual_n, **metrics, 'elapsed_time_sec': elapsed}])
            row_df.to_csv(results_file, mode='a', header=not results_file.exists(), index=False)